In [27]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from datasets import load_dataset
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [11]:
def predict_tcn(model, X_last, n_steps, future_cov=None):
    """
    Prediction recursive pour TCN
    
    Parameters
    ----------
    model : keras model
    X_last : dernière fenêtre (1, seq_len, n_features)
    n_steps : nombre de pas à prédire
    future_cov : covariables futures (n_steps, n_cov) ou None
    
    Returns
    -------
    preds : (n_steps,1)
    """

    X_window = X_last.copy()
    preds = []

    seq_len = X_window.shape[1]

    for step in range(n_steps):

        # prediction
        y_pred = model.predict(X_window, verbose=0)[0,0]
        preds.append(y_pred)

        # shift de la fenêtre
        X_window = np.roll(X_window, -1, axis=1)

        # update target
        X_window[0, -1, 0] = y_pred

        # si on connait les covariables futures
        if future_cov is not None:
            n_cov = future_cov.shape[1]
            X_window[0, -1, 1:1+n_cov] = future_cov[step]

        # sinon on laisse les anciennes valeurs (ou zeros)

    return np.array(preds).reshape(-1,1)

In [12]:
def create_sequences_TCN(X, y, seq_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len])
    return np.array(X_seq), np.array(y_seq).reshape(-1,1)

In [25]:
def fit_tcn(
    X_train, y_train,
    X_val=False, y_val=False,
    X_test=False, y_test=False,
    hidden_channels=[25,25,25],
    kernel_size=3,
    epochs=50,
    batch_size=32,
    lr=0.001,
    loss="mse",
    patience=10,
    verbose=1,
    seq_len=30
):

    # scalers séparés
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    y_train_scaled = scaler_y.fit_transform(y_train.to_numpy().reshape(-1,1))

    if type(X_val) != bool:
        X_val_scaled = scaler_X.transform(X_val)
        y_val_scaled = scaler_y.transform(y_val.to_numpy().reshape(-1,1))

    if type(X_test) != bool:
        X_test_scaled = scaler_X.transform(X_test)

    # création des séquences
    X_train_seq, y_train_seq = create_sequences_TCN(
        X_train_scaled, y_train_scaled, seq_len
    )

    if type(X_val) != bool:
        X_val_seq, y_val_seq = create_sequences_TCN(
            X_val_scaled, y_val_scaled, seq_len
        )

    n_features = X_train_seq.shape[2]

    # modèle TCN
    inputs = layers.Input(shape=(seq_len, n_features))
    x = inputs

    for i, h in enumerate(hidden_channels):
        x = layers.Conv1D(
            filters=h,
            kernel_size=kernel_size,
            padding="causal",
            dilation_rate=2**i
        )(x)
        x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling1D()(x)
    outputs = layers.Dense(1)(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=loss
    )

    # callbacks
    callbacks = []
    if type(X_val) != bool:
        early_stop = EarlyStopping(
            monitor="val_loss",
            patience=patience,
            restore_best_weights=True
        )

        reduce_lr = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=patience//2,
            min_lr=1e-6
        )

        callbacks = [early_stop, reduce_lr]

    # entraînement
    model.fit(
        X_train_seq,
        y_train_seq,
        validation_data=(X_val_seq, y_val_seq) if type(X_val) != bool else None,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        shuffle=False,
        verbose=verbose
    )

    # dernière fenêtre connue
    if type(X_val) != bool:
        X_last = X_val_seq[-1:]
    else:
        X_last = X_train_seq[-1:]

    results = {
        "modele_fit": model,
        "scaler_X": scaler_X,
        "scaler_y": scaler_y
    }

    # prédiction seulement si y_test existe
    if type(y_test) != bool:
    
        n_steps = len(y_test)
    
        # utilisation des covariables futures si disponibles
        if type(X_test) != bool:
            future_cov = X_test_scaled
        else:
            future_cov = None
    
        preds_scaled = predict_tcn(
            model,
            X_last,
            n_steps=n_steps,
            future_cov=future_cov
        )
    
        preds = scaler_y.inverse_transform(preds_scaled)
    
        y_test_real = y_test.to_numpy().reshape(-1,1)
    
        rmse = np.sqrt(mean_squared_error(y_test_real, preds))
        mae = mean_absolute_error(y_test_real, preds)
    
        results["rmse"] = rmse
        results["mae"] = mae

    return results

# Exemple d'utilisation

In [23]:
# Exemple : dataset tourisme (fictive local)

dataset = load_dataset("zaai-ai/time_series_dataset_residuals")
# Choisir le split 'train' (ou 'test')
train_split = dataset['train']

# Convertir ce split en DataFrame
df = train_split.to_pandas()

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df = df.set_index('Date')
df = df.groupby("Date").mean(numeric_only=True) # on aggrege les doublons

# Target
y = df['residuals'][:1000]

# proportions
train_frac = 0.6
val_frac = 0.2
test_frac = 0.2

n_total = len(y)
# print(n_total) # DEBUG

# indices de split
train_end = int(n_total * train_frac)
val_end   = int(n_total * (train_frac + val_frac))

# Splits
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

# on normalise après le split pour éviter le leakage
mean_train = train.mean()
std_train = train.std()

train = (train - mean_train) / std_train
val   = (val - mean_train) / std_train
test  = (test - mean_train) / std_train

def regularize(series):
    return series.reindex(
        pd.date_range(series.index.min(), series.index.max(), freq="MS")
    ).interpolate()

print(df.index.duplicated().sum())

train = regularize(train)
val = regularize(val)
test = regularize(test)

train_val = pd.concat([train, val]).sort_index()

# # supprimer les doublons (si présents)
# # Vérifie les doublons
# print(train.index.duplicated().sum())
# print(val.index.duplicated().sum())
# print(train_val.index.duplicated().sum())
# print(test.index.duplicated().sum())
# train = train[~train.index.duplicated(keep='first')]
# val = val[~val.index.duplicated(keep='first')]
# train_val = train_val[~train_val.index.duplicated(keep='first')]
# test = test[~test.index.duplicated(keep='first')]

# print(train.index.is_monotonic_increasing)
# print(val.index.is_monotonic_increasing)
# print(train_val.index.is_monotonic_increasing)
# print(test.index.is_monotonic_increasing)
# # on trie par date et on réindexe en attribuant une fréquence

print(train.isna().sum())
print(val.isna().sum())
print(train_val.isna().sum())
print(test.isna().sum())

0
0
0
0
0


In [24]:
# Covariables dynamiques
X = df[['CPI', 'Inflation_Rate', 'GDP']][:1000]

# On standardise
X = (X - X.mean()) / X.std()

X_full = pd.concat([y, X], axis=1)

# Train
X_train = X_full.iloc[:train_end]

# Validation
X_val = X_full.iloc[train_end:val_end]

# Test
X_test = X.iloc[val_end:].astype(float)

# Train-Val
X_train_val = pd.concat([X_train, X_val])

# on standardise y
y = (y - y.mean()) / y.std()

# création des ensembles
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

train_val = pd.concat([train, val])

In [29]:
# entraînement + évaluation
seq_len = 30
resultat = fit_tcn(X_train, train, X_val=False, y_val=val, X_test=False, y_test=test, seq_len=seq_len)

print(resultat['rmse'], resultat['mae'])

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - loss: 0.0547
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0318
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0265 
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0252
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0234
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0224
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0225
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0229
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0228
Epoch 10/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0222 
Epoch 11/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0213
Epoch 12/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - loss: 0.0207
Epoch 13/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0203
Epoch 14/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0200
Epoch 15/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0196
Epoch 16/50
3/3 ━━━━━━━━━━━━━━━━

In [37]:
def predict_tcn_simple(resultat, der, X_der, n_step, seq_len, X_test=False):
    if type(X_test) != bool:
        # créer un placeholder
        X_test_full = X_test.copy()
        X_test_full.insert(0, "residuals", 0)
        # utilisation des covariables futures
        future_cov = resultat['scaler_X'].transform(X_test_full)
        # enlever la target
        future_cov = future_cov[:,1:]
    else:
        future_cov = None
    
    # on retrouve X_last
    X_val_scaled = resultat['scaler_X'].transform(X_der)
    y_val_scaled = resultat['scaler_y'].transform(der.to_numpy().reshape(-1,1))
    X_last = create_sequences_TCN(X_val_scaled, y_val_scaled, seq_len)[0][-1:]
    
    preds_scaled = predict_tcn(
        resultat['modele_fit'],
        X_last,
        n_steps=n_steps,
        future_cov=future_cov
    )

    return resultat['scaler_y'].inverse_transform(preds_scaled)

In [38]:
predictions = predict_tcn_simple(resultat, val, X_val, len(test), seq_len, X_test=X_test)
predictions[:10]

array([[-0.05912785],
       [-0.06566104],
       [-0.05661918],
       [-0.07014253],
       [-0.07236603],
       [-0.04448615],
       [-0.04736609],
       [-0.03606583],
       [-0.02283405],
       [-0.01744854]], dtype=float32)